# 6단계 실험: WordPress 발행 (Review & Publish)

1~4단계로 만든 초안 + SEO 메타를 WordPress REST API 로 업로드합니다. **기본은 draft** — 사람이 WP 어드민에서 확인 후 publish 로 전환합니다.

**준비**
- WordPress 5.6+ (Application Passwords 내장)
- WP 어드민 → Users → 본인 → **Application Passwords** 발급
- `.env` 에 `WORDPRESS_URL`, `WORDPRESS_USERNAME`, `WORDPRESS_APP_PASSWORD` 채우기
- `WORDPRESS_URL` 형식: `https://yoursite.com` (끝 슬래시 없어도 됨)

## 0. 환경 설정

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY",
         "WORDPRESS_URL", "WORDPRESS_USERNAME", "WORDPRESS_APP_PASSWORD"):
    val = os.getenv(k)
    show = "OK" if val else "MISSING"
    if k == "WORDPRESS_URL":
        show = val or "MISSING"
    print(f"{k}: {show}")

## 1. WP 연결 헬스체크

Application Password 가 유효한지 가벼운 GET 으로 확인.

In [ ]:
import requests
from requests.auth import HTTPBasicAuth

site = os.environ["WORDPRESS_URL"].rstrip("/")
auth = HTTPBasicAuth(os.environ["WORDPRESS_USERNAME"], os.environ["WORDPRESS_APP_PASSWORD"])

r = requests.get(f"{site}/wp-json/wp/v2/users/me", auth=auth, timeout=15)
print("status:", r.status_code)
if r.ok:
    me = r.json()
    print(f"logged in as: {me['name']} (id={me['id']}, roles={me.get('roles')})")
else:
    print("body:", r.text[:300])

## 2. 초안 + SEO 메타 준비

옵션 A: 풀 파이프라인 신규 실행. 옵션 B: 기존 `data/drafts/*.md` 재사용 (비용 0).

In [ ]:
TOPIC = "외국인을 위한 한국 길거리 음식 추천"

# --- 옵션 A: 신규 생성 ---
from llm_jacky.research import run_research
from llm_jacky.rag import index_research, _vector_store
from llm_jacky.writing import write_draft
from llm_jacky.seo import generate_seo

if len(_vector_store().get().get("ids", [])) == 0:
    index_research(run_research(TOPIC, max_results=4))

draft = write_draft(TOPIC, k=4)
meta = generate_seo(TOPIC, draft.draft)
DRAFT_MD = draft.draft

# --- 옵션 B: 기존 파일 재사용 ---
# DRAFT_MD = (PROJECT_ROOT / "data" / "drafts" / "<your-slug>.md").read_text(encoding="utf-8")
# from llm_jacky.seo import SeoMeta
# meta = SeoMeta(title="...", meta_description="...", tags=["..."], slug="...")

print("title :", meta.title)
print("slug  :", meta.slug)
print("tags  :", meta.tags)
print("chars :", len(DRAFT_MD))

## 3. HTML 변환 미리보기 (dry-run)

WP 에 보내기 전에 마크다운 → HTML 결과를 한 번 확인합니다.

In [ ]:
from llm_jacky.publish import render_html

html = render_html(DRAFT_MD)
print(html[:1200], "...")

## 4. 초안(Draft) 으로 업로드

`status="draft"` 가 기본. 이 셀은 **외부 가시 액션**이지만 발행되지 않은 상태이므로 안전합니다.

In [ ]:
from llm_jacky.publish import publish_post

result = publish_post(
    title=meta.title,
    content_md=DRAFT_MD,
    excerpt=meta.meta_description,
    slug=meta.slug,
    tags=meta.tags,
    status="draft",
)

print("post id   :", result.id)
print("status    :", result.status)
print("public url:", result.url, "(draft 이라 비공개)")
print("edit url  :", result.edit_url)
print("\n-> 어드민에서 위 edit url 을 열어 검토하세요.")

## 5. (선택) 발행으로 전환

**주의**: 이 셀을 실행하면 **공개 발행**됩니다. WP 어드민에서 직접 publish 누르는 것이 더 안전합니다.

코드로 토글하고 싶다면 아래 셀 한 줄 주석 해제.

In [ ]:
# 공개 발행으로 전환 (정말 발행할 때만 주석 해제)
# r = requests.post(
#     f"{site}/wp-json/wp/v2/posts/{result.id}",
#     json={"status": "publish"},
#     auth=auth,
#     timeout=20,
# )
# r.raise_for_status()
# print("now status:", r.json()["status"], "| link:", r.json()["link"])

## 6. 정리: 테스트 글 삭제 (옵션)

실험 후 테스트 게시물을 휴지통으로 보냅니다 (`force=True` 면 영구삭제).

In [ ]:
# r = requests.delete(
#     f"{site}/wp-json/wp/v2/posts/{result.id}",
#     auth=auth,
#     timeout=15,
# )
# print("delete:", r.status_code)